In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
data_raw   = pd.read_csv("../data/input_data/data_imputed.csv")
print(data_raw.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133887 entries, 0 to 133886
Data columns (total 16 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   ID_VICTIMA                       133887 non-null  object
 1   ORIGEN_REPORTE                   133887 non-null  object
 2   FECHA_NACIMIENTO                 133887 non-null  object
 3   SEXO                             133887 non-null  object
 4   FECHA_DESAPARICION               133887 non-null  object
 5   FECHA_REGISTRO                   133887 non-null  object
 6   ESTATUS_VICTIMA                  133887 non-null  object
 7   CVE_ENT                          133887 non-null  int64 
 8   ENTIDAD                          133887 non-null  object
 9   CVE_MUN                          133887 non-null  int64 
 10  MUNICIPIO                        133887 non-null  object
 11  SEXO_MAP                         133887 non-null  int64 
 12  ESTATUS_MAP     

In [4]:
# EDAD = FECHA_DESAPARICION - FECHA_NACIMIENTO Luego Pasar a Rango
data_raw["EDAD"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce') - pd.to_datetime(data_raw["FECHA_NACIMIENTO"], errors='coerce')
data_raw["EDAD"] = data_raw["EDAD"].dt.days // 365

for i in data_raw["EDAD"].index:
    if data_raw["EDAD"][i] < 12:
        data_raw["EDAD"][i] = "Niño"
    elif data_raw["EDAD"][i] < 18:
        data_raw["EDAD"][i] = "Adolescente"
    elif data_raw["EDAD"][i] < 30:
        data_raw["EDAD"][i] = "Joven"
    elif data_raw["EDAD"][i] < 60:
        data_raw["EDAD"][i] = "Adulto"
    else:
        data_raw["EDAD"][i] = "Adulto Mayor"
        
print(data_raw["EDAD"].value_counts())

C:\Users\Nancy\AppData\Local\Temp\ipykernel_14864\3783820525.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data_raw["EDAD"][i] = "Joven"
C:\Users\Nancy\AppData\Local\Temp\ipykernel_14864\3783820525.py:11: SettingWithCopyWarning: 
A va

EDAD
Adulto          66784
Joven           55622
Adolescente      5068
Niño             3376
Adulto Mayor     3037
Name: count, dtype: int64


In [5]:
# Extraer de la Fecha de Desaparicion el año y el mes y dia de la semana
data_raw["AÑO_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.year
data_raw["MES_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.month
data_raw["DIA_SEMANA_DESAPARICION"] = pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce').dt.dayofweek 

In [6]:
# Restar la fecha de desaparicion con la de reporte para saber si fue inmediato o no
data_raw["TIEMPO_REPORTE"] = pd.to_datetime(data_raw["FECHA_REGISTRO"], errors='coerce') - pd.to_datetime(data_raw["FECHA_DESAPARICION"], errors='coerce')
data_raw["TIEMPO_REPORTE"] = data_raw["TIEMPO_REPORTE"].dt.days

for i in data_raw["TIEMPO_REPORTE"].index:
    if data_raw["TIEMPO_REPORTE"][i] <= 1:
        data_raw["TIEMPO_REPORTE"][i] = "Inmediato"
    elif data_raw["TIEMPO_REPORTE"][i] <= 7:
        data_raw["TIEMPO_REPORTE"][i] = "Rapido"
    elif data_raw["TIEMPO_REPORTE"][i] <= 30:
        data_raw["TIEMPO_REPORTE"][i] = "Tardio"
    else:
        data_raw["TIEMPO_REPORTE"][i] = "Muy Tardio"

print(data_raw["TIEMPO_REPORTE"].value_counts())

C:\Users\Nancy\AppData\Local\Temp\ipykernel_14864\1247707725.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data_raw["TIEMPO_REPORTE"][i] = "Inmediato"
C:\Users\Nancy\AppData\Local\Temp\ipykernel_14864\1247707725.py:7: SettingWithCopyWa

TIEMPO_REPORTE
Inmediato     128522
Muy Tardio      2201
Rapido          2127
Tardio          1037
Name: count, dtype: int64


In [ ]:
dfFinal = data_raw[['EDAD', 'SEXO', 'AÑO_DESAPARICION', 'MES_DESAPARICION', 'TIEMPO_REPORTE', 
                    'ESTATUS_VICTIMA', 'ENTIDAD']]

transacciones = []

for _, row in dfFinal.iterrows():
    trans = [
        f"EDAD_{row['EDAD']}",
        f"SEXO_{row['SEXO']}",
        f"AÑO_{row['AÑO_DESAPARICION']}",
        f"MES_{row['MES_DESAPARICION']}",
        f"TIEMPO_{row['TIEMPO_REPORTE']}",
        f"ESTATUS_{row['ESTATUS_VICTIMA']}",
        f"ENTIDAD_{row['ENTIDAD']}"
    ]
    transacciones.append(trans)

print(transacciones[:5])

[['EDAD_Joven', 'SEXO_CONFIDENCIAL', 'AÑO_2012', 'MES_8', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_DURANGO'], ['EDAD_Adulto', 'SEXO_HOMBRE', 'AÑO_2025', 'MES_9', 'TIEMPO_Inmediato', 'ESTATUS_DESAPARECIDA', 'ENTIDAD_SINALOA'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2020', 'MES_1', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_CIUDAD DE MÉXICO'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2023', 'MES_2', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_BAJA CALIFORNIA'], ['EDAD_Adulto', 'SEXO_CONFIDENCIAL', 'AÑO_2023', 'MES_2', 'TIEMPO_Inmediato', 'ESTATUS_CONFIDENCIAL', 'ENTIDAD_BAJA CALIFORNIA']]


In [20]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transacciones).transform(transacciones)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(df_encoded.head(5))

   AÑO_1961  AÑO_1962  AÑO_1964  AÑO_1967  AÑO_1968  AÑO_1969  AÑO_1970  \
0     False     False     False     False     False     False     False   
1     False     False     False     False     False     False     False   
2     False     False     False     False     False     False     False   
3     False     False     False     False     False     False     False   
4     False     False     False     False     False     False     False   

   AÑO_1971  AÑO_1972  AÑO_1973  ...  MES_8  MES_9  SEXO_CONFIDENCIAL  \
0     False     False     False  ...   True  False               True   
1     False     False     False  ...  False   True              False   
2     False     False     False  ...  False  False               True   
3     False     False     False  ...  False  False               True   
4     False     False     False  ...  False  False               True   

   SEXO_HOMBRE  SEXO_INDETERMINADO  SEXO_MUJER  TIEMPO_Inmediato  \
0        False               False       F

In [22]:
rasgosFrecuentes = apriori(df_encoded, min_support=0.1, use_colnames=True)
print("Rasgos Frecuentes Totales:", rasgosFrecuentes.shape[0])

Rasgos Frecuentes Totales: 66


In [23]:
rules = association_rules(
    rasgosFrecuentes,
    metric="confidence",
    min_threshold=0.5
)

rules = rules[ (rules['confidence'] >= 0.5) & (rules['lift'] > 1)]

rules = rules.sort_values(by='lift', ascending=False)

for i, row in rules.iterrows():
    print(f"{set(row['antecedents'])} → {set(row['consequents'])}")
    print(f"support: {row['support']:.3f}, confidence: {row['confidence']:.3f}, lift: {row['lift']:.3f}")
    print("-----")

{'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL'} → {'SEXO_CONFIDENCIAL'}
support: 0.367, confidence: 1.000, lift: 2.724
-----
{'SEXO_CONFIDENCIAL', 'EDAD_Adulto'} → {'ESTATUS_CONFIDENCIAL'}
support: 0.214, confidence: 1.000, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL', 'EDAD_Adulto'} → {'SEXO_CONFIDENCIAL'}
support: 0.214, confidence: 1.000, lift: 2.724
-----
{'SEXO_CONFIDENCIAL'} → {'ESTATUS_CONFIDENCIAL', 'EDAD_Adulto'}
support: 0.214, confidence: 0.582, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL'} → {'SEXO_CONFIDENCIAL', 'EDAD_Adulto'}
support: 0.214, confidence: 0.582, lift: 2.724
-----
{'SEXO_CONFIDENCIAL', 'EDAD_Joven'} → {'ESTATUS_CONFIDENCIAL', 'TIEMPO_Inmediato'}
support: 0.153, confidence: 1.000, lift: 2.724
-----
{'ESTATUS_CONFIDENCIAL', 'EDAD_Joven'} → {'TIEMPO_Inmediato', 'SEXO_CONFIDENCIAL'}
support: 0.153, confidence: 1.000, lift: 2.724
-----
{'TIEMPO_Inmediato', 'SEXO_CONFIDENCIAL', 'EDAD_Joven'} 